<a href="https://colab.research.google.com/github/siddharthsinh-dev/iu-bsc-thesis-llm-comparison/blob/main/notebooks/exp1_c_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Thesis**: A Comparative Study of Large Language Models for Financial Sentiment Analysis and Their Predictive Potential for Short-Term Stock Price Movement

**Component: Experiment 1** - Model C : Mistral (7B Instruct v0.3)

**Description:** This notebook evaluates Mistral 7B on the Financial PhraseBank dataset under zero-shot conditions. It generates sentiment predictions and computes accuracy, precision, recall, F1-score (macro-averaged), and confusion matrix.

Select T4 GPU as runtime.

In [ ]:
# Step 1 — Install bitsandbytes (restart required after this)

!pip install -q -U bitsandbytes accelerate

Go to runtime -> restart this session again -> then run cell 1 and run cell 2

In [ ]:
# Import required libraries — Experiment 1c: Mistral

import pandas as pd
import numpy as np
import torch
import warnings
import gc

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings("ignore")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detected    : {torch.cuda.get_device_name(0)}")

Add the API Key from Hugging Face using "Add New Secret" in Google Colab

In [ ]:
# Connect to Hugging Face and load Financial PhraseBank

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

print("Hugging Face login successful.")

# Load Financial PhraseBank — full dataset
file_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    filename="sentences_50agree/train-00000-of-00001.parquet",
    repo_type="dataset",
    revision="0dd3028d70cbd18ded8887e65e83343b03a50482",
    token=hf_token
)

df_fpb = pd.read_parquet(file_path)

label_map = {0: "negative", 1: "neutral", 2: "positive"}
df_fpb["sentiment"] = df_fpb["label"].map(label_map)

true_labels = df_fpb["sentiment"].tolist()
texts = df_fpb["sentence"].tolist()

print(f"\nFinancial PhraseBank loaded.")
print(f"Total sentences : {len(df_fpb)}")
print(f"Label distribution:")
print(df_fpb["sentiment"].value_counts())

In [ ]:
# Load Mistral 7B Instruct v0.3

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Mistral 7B Instruct v0.3...")

mistral_tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    token=hf_token
)

mistral_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("Mistral 7B loaded successfully.")

In [ ]:
# Define Mistral classifier and run on full dataset

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

def clean_label(label):
    label = label.strip().lower()
    if "positive" in label:
        return "positive"
    elif "negative" in label:
        return "negative"
    elif "neutral" in label:
        return "neutral"
    else:
        return "neutral"

def classify_mistral(text):
    prompt = f"""You are a financial sentiment classifier.

Classify the sentiment of this financial text into exactly one word: positive, negative, or neutral.

Text: {text}

Sentiment:"""

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(mistral_model.device)

    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    generated = mistral_tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )
    return clean_label(generated)

# Run on full dataset
mistral_preds = []
total = len(texts)

print(f"Running Mistral on {total} sentences...")

for i, text in enumerate(texts):
    label = classify_mistral(text)
    mistral_preds.append(label)

    if (i + 1) % 100 == 0:
        print(f"  Progress: {i+1}/{total}")

print(f"\nDone. Total predictions: {len(mistral_preds)}")
print(f"\nPrediction distribution:")
print(pd.Series(mistral_preds).value_counts())

In [ ]:
# Evaluate Mistral performance on full dataset

labels_order = ["positive", "negative", "neutral"]

acc  = accuracy_score(true_labels, mistral_preds)
prec = precision_score(true_labels, mistral_preds, average="macro", labels=labels_order)
rec  = recall_score(true_labels, mistral_preds, average="macro", labels=labels_order)
f1   = f1_score(true_labels, mistral_preds, average="macro", labels=labels_order)

print("=" * 45)
print("Mistral 7B — Experiment 1 Results")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 45)

print("\nDetailed Classification Report:")
print(classification_report(true_labels, mistral_preds, labels=labels_order))

In [ ]:
# Save Mistral results

mistral_scores = {
    "model": "Mistral 7B",
    "accuracy": round(acc, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "f1_score": round(f1, 4)
}

df_fpb["mistral_pred"] = mistral_preds

print("Mistral 7B — Results Summary")
print(pd.DataFrame([mistral_scores]))

In [ ]:
# Save Mistral predictions to Google Drive

from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/Thesis_Data", exist_ok=True)

df_fpb.to_csv("/content/drive/MyDrive/Thesis_Data/exp1_mistral_preds.csv", index=False)

print("Mistral predictions saved to Google Drive.")

In [ ]:
# Confusion Matrix — Mistral 7B Experiment 1

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels_order = ["positive", "negative", "neutral"]

cm = confusion_matrix(true_labels, mistral_preds, labels=labels_order)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_order,
            yticklabels=labels_order)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Mistral 7B — Confusion Matrix (Experiment 1)", fontsize=12)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/Thesis_Data/fig_cm_mistral.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Confusion matrix saved to Google Drive.")